# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json# Define the Croissant schema URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadata from the schemadataset = mlc.Dataset(croissant_url)metadata = dataset.metadataprint(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data OverviewReview available record sets, fields, and their IDs.

In [ ]:
# Display all available record sets and their fields by @idrecord_sets = list(dataset.list_record_sets())print("Available Record Sets (by @id):")for r in record_sets:    print(f"- @id: {r['@id']}, name: {r.get('name')}")# For each record set, print available fields and their @idfor r in record_sets:    print(f"\nRecord set: {r.get('name', 'N/A')} (@id: {r['@id']})")    fields = r.get('field', [])    if isinstance(fields, dict):        fields = [fields]    print("Fields:")    for f in fields:        # Each field is a dict like {'@id': ..., 'name': ...}        f_id = f.get('@id', str(f))        f_name = f.get('name', 'N/A')        print(f" - @id: {f_id}, name: {f_name}")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Choose record sets by @id for extraction (replace with actual found values if needed)record_set_ids = [r['@id'] for r in list(dataset.list_record_sets())]dataframes = {}for rs_id in record_set_ids:    # Load all records for this record set    records = list(dataset.records(record_set=rs_id))    # Convert list of dicts to dataframe    if records:        dataframes[rs_id] = pd.DataFrame(records)# Display available record sets and their columnsfor rs_id, df in dataframes.items():    print(f"Record set @id: {rs_id}")    print(f"Columns: {df.columns.tolist()}")    display(df.head())

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping by key attributes.

In [ ]:
# For demonstration, select the first available record set and attempt to analyze numeric fieldsif dataframes:    selected_record_set_id = list(dataframes.keys())[0]    df = dataframes[selected_record_set_id]    # Try to automatically select a numeric field    numeric_cols = df.select_dtypes(include=['float', 'float64', 'int', 'int64']).columns.tolist()    if not numeric_cols:        # Attempt to convert columns to numeric if dataset was loaded as strings        for col in df.columns:            try:                df[col] = pd.to_numeric(df[col])            except Exception:                continue        numeric_cols = df.select_dtypes(include=['float', 'float64', 'int', 'int64']).columns.tolist()    if numeric_cols:        numeric_field = numeric_cols[0]        threshold = df[numeric_field].quantile(0.75)  # use 75th percentile as example threshold        filtered_df = df[df[numeric_field] > threshold].copy()        print(f"Filtered records with {numeric_field} > {threshold}:")        display(filtered_df.head())        # Normalization        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()        print(f"Normalized {numeric_field} for filtered records:")        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())        # Try grouping by next non-numeric field, if present        group_fields = [c for c in df.columns if c not in numeric_cols]        group_field = group_fields[0] if group_fields else None        if group_field:            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)            print(f"Grouped data by {group_field} (using field '@id'): {group_field}")            display(grouped_df.head())        else:            print("No non-numeric group field available for grouping.")    else:        print("No numeric fields detected for EDA.")else:    print("No dataframes loaded for EDA.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Visualize distribution of a numeric field, if availableif dataframes and numeric_cols:    plt.figure(figsize=(8, 5))    sns.histplot(df[numeric_field].dropna(), kde=True)    plt.title(f"Distribution of {numeric_field}")    plt.xlabel(numeric_field)    plt.ylabel("Frequency")    plt.show()    # Boxplot by group field, if available    if group_field:        plt.figure(figsize=(10, 5))        sns.boxplot(x=group_field, y=numeric_field, data=df)        plt.title(f"{numeric_field} by {group_field}")        plt.xticks(rotation=45)        plt.show()

## 6. ConclusionSummarize key findings and observations from the dataset exploration.- The dataset contains multiple record sets with structured fields, allowing detailed protein characterization.- Numeric and categorical fields can be analyzed and visualized using standard data science workflows.- Use the record set and field `@id` values for precise, reproducible data extraction and processing.For further analysis, refer to the Croissant schema for additional relationships and documentation of each field.